# Colab で 3DGS を実行する
### 事前準備
- この Notebook のカーネルを Colab の GPU の Python3 にすること

## Google Drive へマウント
- 注：セッション内で1回だけ実行

In [1]:
# 現在のセッションをGoogle Driveにマウントする
from google.colab import drive
drive.mount('/content/drive')    # "/content/drive"以下は Google Driveのルートディレクトリに対応

# Drive 内に作業用フォルダを作成
!mkdir -p /content/drive/MyDrive/Colab/3DGS

# 上記作業フォルダに簡単にアクセスするためのシンボリックリンクを作成
!ln -s /content/drive/MyDrive/Colab/3DGS /content/3DGS

Mounted at /content/drive


## 3DGS のリポジトリを Google Drive 上にクローン
- 注：全体で1回だけ実行（Google Drive 上にファイルができたらそれ以降は実施不要）

In [2]:
!git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting.git /content/3DGS/gaussian-splatting

Cloning into '/content/3DGS/gaussian-splatting'...
remote: Enumerating objects: 1053, done.
remote: Total 1053 (delta 0), reused 0 (delta 0), pack-reused 1053 (from 1)
Receiving objects: 100% (1053/1053), 78.72 MiB | 18.46 MiB/s, done.
Resolving deltas: 100% (592/592), done.
Updating files: 100% (64/64), done.
Submodule 'SIBR_viewers' (https://gitlab.inria.fr/sibr/sibr_core.git) registered for path 'SIBR_viewers'
Submodule 'submodules/diff-gaussian-rasterization' (https://github.com/graphdeco-inria/diff-gaussian-rasterization.git) registered for path 'submodules/diff-gaussian-rasterization'
Submodule 'submodules/fused-ssim' (https://github.com/rahul-goel/fused-ssim.git) registered for path 'submodules/fused-ssim'
Submodule 'submodules/simple-knn' (https://gitlab.inria.fr/bkerbl/simple-knn.git) registered for path 'submodules/simple-knn'
Cloning into '/content/drive/.shortcut-targets-by-id/1-5Lb51YweqU3-UdVdzNzc5wKbd9O16L4/Colab/3DGS/gaussian-splatting/SIBR_viewers'...
remote: Enumerati

## PIP パッケージのインストール
- 注：セッションで1回だけ実行

In [3]:
# 基本パッケージのインストール
!pip install plyfile tqdm

# サブモジュールのビルド（重要）
# これにより CUDA カーネルがビルドされます
!pip install -e /content/3DGS/gaussian-splatting/submodules/diff-gaussian-rasterization
!pip install -e /content/3DGS/gaussian-splatting/submodules/simple-knn

# カメラ位置推定ツールのインストール
!sudo apt-get update
!sudo apt-get install -y colmap

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 3.7 MB/s eta 0:00:00
Obtaining file:///content/3DGS/gaussian-splatting/submodules/diff-gaussian-rasterization
  Preparing metadata (setup.py) ... done
  Running setup.py develop for diff_gaussian_rasterization
Obtaining file:///content/3DGS/gaussian-splatting/submodules/simple-knn
  Preparing metadata (setup.py) ... done
  Running setup.py develop for simple_knn
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]               
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,608 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu 

## 学習する

### 画像ファイルの準備
1. Google Drive 上に、`適当なフォルダ名/input/`フォルダを作成し、その下にさまざまな角度から撮影した画像ファイルを配置する。
1. 今回は後の処理を統一するため、上記のフォルダに対し、`input_data`という名前のシンボリックリンクを通す。

In [4]:
# 第2引数に、「適当なフォルダ名」を指定する（配下には「input/画像ファイル」の構成で画像を配置する）
# 第3引数「/content/input_data」は変更不要
!ln -s /content/3DGS/input_dirName /content/input_data

In [5]:
# 画像からカメラ位置を計算する（COLMAPを使用）

import os
# Qtに「画面がない（offscreen）」ことを伝える
os.environ['QT_QPA_PLATFORM'] = 'offscreen'

!xvfb-run -a python /content/3DGS/gaussian-splatting/convert.py -s /content/input_data/ --camera PINHOLE

QStandardPaths: XDG_RUNTIME_DIR not set, defaulting to '/tmp/runtime-root'

Feature extraction

Processed file [1/27]
  Name:            frame_00000.jpg
  Dimensions:      1920 x 1080
  Camera:          #1 - PINHOLE
  Focal Length:    2304.00px
  Features:        2551
Processed file [2/27]
  Name:            frame_00001.jpg
  Dimensions:      1920 x 1080
  Camera:          #1 - PINHOLE
  Focal Length:    2304.00px
  Features:        1716
Processed file [3/27]
  Name:            frame_00002.jpg
  Dimensions:      1920 x 1080
  Camera:          #1 - PINHOLE
  Focal Length:    2304.00px
  Features:        1455
Processed file [4/27]
  Name:            frame_00003.jpg
  Dimensions:      1920 x 1080
  Camera:          #1 - PINHOLE
  Focal Length:    2304.00px
  Features:        1452
Processed file [5/27]
  Name:            frame_00004.jpg
  Dimensions:      1920 x 1080
  Camera:          #1 - PINHOLE
  Focal Length:    2304.00px
  Features:        1581
Processed file [6/27]
  Name:          

### 学習
- 以下を実行すると、`output/<ランダムなID>/point_cloud/iteration_iter数/point_cloud.ply`に 3DGS モデルが出力される
- iteration 数は適当に変更する（7000-30000）

In [6]:
# 学習する
!python /content/3DGS/gaussian-splatting/train.py -s /content/input_data/ \
    --iterations 7000 \
    --densify_until_iter 5000 \
    --test_iterations 7000 \
    --save_iterations 7000

2026-05-03 15:47:54.794892: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Optimizing 
Output folder: ./output/d4beeb36-6 [03/05 15:47:58]
Reading camera 27/27 [03/05 15:47:58]
Converting point3d.bin to .ply, will happen only the first time you open the scene. [03/05 15:47:58]
Loading Training Cameras [03/05 15:47:59]
[ INFO ] Encountered quite large input images (>1.6K pixels width), rescaling to 1.6K.
 If this is not desired, please explicitly specify '--resolution/-r' as 1 [03/05 15:47:59]
Loading Test Cameras [03/05 15:48:01]
Number of points at initialisation :  1807 [03/05 15:48:01]
Training progress: 100% 7000/7000 [11:31<00:00, 10.12it/s, Loss=0.0231115, Depth Loss=0.0000000]

[ITER 7000] Evaluating train: L1 0.01085354443639517 PSNR 35.

In [7]:
# 一応ドライブ上にもコピーしておく
!cp -r /content/output /content/3DGS/

## モデルの表示
- `render.py` はまだ試していない（ローカルでGPUが必要？）

### ウェブサービスを使う場合
- 例えば以下のサービスにファイルをアップロードすると、ブラウザ上でモデルを確認できる
    - https://superspl.at/editor
        - どこにもモデルが無いかのように見えても、適当に見る角度を変えていると見つかる場合がある

### Blender とアドオンで見る場合
- https://github.com/Kiri-Innovation/3dgs-render-blender-addon